# Document Question Answering System (RAG)
**Archie Saxena**

A Retrieval-Augmented Generation pipeline: retrieve relevant chunks from a custom document, then generate an answer grounded in that retrieved context — not the model's parametric memory.

This notebook is self-contained: it tries the full model-based pipeline (sentence-transformers embeddings + FAISS + a local flan-t5 generator) and automatically falls back to a dependency-light TF-IDF + extractive pipeline if it can't reach Hugging Face (e.g. no internet, or a locked-down sandbox). Both paths implement the same 7 RAG stages.

## 1. Setup

In [1]:
import os, re, glob
import numpy as np
from pathlib import Path

DOCS_FOLDER = "sample_docs"
CHUNK_SIZE = 120      # words per chunk
CHUNK_OVERLAP = 30    # overlap words
TOP_K = 3

## 2. Stage 1-2: Document Ingestion + Chunking
Loads every `.txt`/`.pdf` in `sample_docs/`, then splits into overlapping word-based chunks so a fact split at a chunk boundary still appears whole in the next chunk.

In [2]:
def load_documents(folder):
    docs = {}
    for path in Path(folder).glob("*"):
        if path.suffix.lower() == ".txt":
            docs[path.name] = path.read_text(encoding="utf-8", errors="ignore")
        elif path.suffix.lower() == ".pdf":
            from pypdf import PdfReader
            reader = PdfReader(str(path))
            docs[path.name] = "\n".join((p.extract_text() or "") for p in reader.pages)
    return docs

def chunk_text(text, chunk_size=CHUNK_SIZE, overlap=CHUNK_OVERLAP):
    words = text.split()
    if not words:
        return []
    chunks, step = [], chunk_size - overlap
    for start in range(0, len(words), step):
        piece = words[start:start + chunk_size]
        if not piece:
            break
        chunks.append(" ".join(piece))
        if start + chunk_size >= len(words):
            break
    return chunks

docs = load_documents(DOCS_FOLDER)
chunks = []          # list of chunk strings
chunk_sources = []    # (filename, chunk_id) per chunk
for fname, text in docs.items():
    for i, c in enumerate(chunk_text(text)):
        chunks.append(c)
        chunk_sources.append((fname, i))

print(f"Loaded {len(docs)} document(s), split into {len(chunks)} chunks.")
print(f"\nChunk 0 preview:\n{chunks[0][:300]}...")

Loaded 1 document(s), split into 4 chunks.

Chunk 0 preview:
Retrieval-Augmented Generation (RAG) is an AI architecture that combines information retrieval with text generation. Instead of relying only on the parametric knowledge stored inside a language model's weights, a RAG system retrieves relevant text from an external knowledge source at query time and ...


## 3. Stage 3-4: Embeddings + Vector Store
Tries `sentence-transformers` (all-MiniLM-L6-v2) + FAISS first — this is the production path (see `embed_store.py` in the submitted code). If the model can't be downloaded (no internet), falls back to scikit-learn's TF-IDF as the vector representation, with cosine similarity for search. Both expose the same `embed_query()` / `search()` interface so Stage 5-7 below don't need to know which backend is active.

In [3]:
USE_NEURAL_EMBEDDINGS = False

try:
    from sentence_transformers import SentenceTransformer
    import faiss

    _model = SentenceTransformer("all-MiniLM-L6-v2")
    _emb = _model.encode(chunks, convert_to_numpy=True).astype("float32")
    faiss.normalize_L2(_emb)
    _index = faiss.IndexFlatIP(_emb.shape[1])
    _index.add(_emb)

    def search(query, top_k=TOP_K):
        q = _model.encode([query], convert_to_numpy=True).astype("float32")
        faiss.normalize_L2(q)
        scores, idx = _index.search(q, top_k)
        return [(chunks[i], chunk_sources[i], float(s)) for s, i in zip(scores[0], idx[0]) if i != -1]

    USE_NEURAL_EMBEDDINGS = True
    print("Using neural embeddings: sentence-transformers (all-MiniLM-L6-v2) + FAISS")

except Exception as e:
    print(f"Neural embedding path unavailable ({type(e).__name__}: {e})")
    print("Falling back to TF-IDF + cosine similarity (scikit-learn, no download required).\n")

    from sklearn.feature_extraction.text import TfidfVectorizer
    from sklearn.metrics.pairwise import cosine_similarity

    _vectorizer = TfidfVectorizer(stop_words="english")
    _tfidf_matrix = _vectorizer.fit_transform(chunks)

    def search(query, top_k=TOP_K):
        q_vec = _vectorizer.transform([query])
        sims = cosine_similarity(q_vec, _tfidf_matrix)[0]
        top_idx = sims.argsort()[::-1][:top_k]
        return [(chunks[i], chunk_sources[i], float(sims[i])) for i in top_idx if sims[i] > 0]

    print("Using TF-IDF + cosine similarity")

Neural embedding path unavailable (OSError: We couldn't connect to 'https://huggingface.co' to load the files, and couldn't find them in the cached files.
Check your internet connection or see how to run the library in offline mode at 'https://huggingface.co/docs/transformers/installation#offline-mode'.)
Falling back to TF-IDF + cosine similarity (scikit-learn, no download required).

Using TF-IDF + cosine similarity


## 4. Stage 5-6: Query Processing + Context Retrieval
Quick sanity check — embed a question, retrieve the top-k most relevant chunks, before wiring up generation.

In [4]:
test_results = search("What are the three stages of a RAG pipeline?")
for text, (fname, cid), score in test_results:
    print(f"[{score:.3f}] {fname} chunk#{cid}")
    print(f"  {text[:160]}...\n")

[0.277] sample.txt chunk#0
  Retrieval-Augmented Generation (RAG) is an AI architecture that combines information retrieval with text generation. Instead of relying only on the parametric k...

[0.204] sample.txt chunk#1
  by pairing a retriever with a generator. A typical RAG pipeline has three stages. First, documents are split into smaller chunks and converted into vector embed...

[0.117] sample.txt chunk#2
  generates an answer grounded in that retrieved context. RAG systems are widely used to build chatbots that can answer questions about a company's internal docum...



## 5. Stage 7: Answer Generation
Same fallback logic as embeddings: tries `google/flan-t5-base` (local, no API key, but needs the one-time Hugging Face download) for real grounded generation. If unavailable, falls back to an **extractive** generator — it returns the most relevant sentence(s) from the retrieved chunks rather than a fluently rewritten answer. This is weaker than LLM generation but still fully grounded (never hallucinates), and needs zero downloads.

In [5]:
USE_LLM_GENERATION = False

try:
    from transformers import pipeline as hf_pipeline
    _generator = hf_pipeline("text2text-generation", model="google/flan-t5-base")

    def generate_answer(question, retrieved):
        context = "\n\n".join(f"[{fname}]\n{text}" for text, (fname, cid), score in retrieved)
        prompt = (
            "Answer the question using ONLY the context below. "
            "If the context does not contain the answer, say "
            "'The document does not contain this information.'\n\n"
            f"Context:\n{context}\n\nQuestion: {question}\nAnswer:"
        )
        return _generator(prompt, max_new_tokens=200, do_sample=False)[0]["generated_text"].strip()

    USE_LLM_GENERATION = True
    print("Using LLM generation: google/flan-t5-base")

except Exception as e:
    print(f"LLM generation path unavailable ({type(e).__name__}: {e})")
    print("Falling back to extractive generation (best-matching sentences, no download required).\n")

    import re as _re

    def generate_answer(question, retrieved):
        if not retrieved:
            return "The document does not contain this information."
        q_words = set(_re.findall(r"\w+", question.lower()))
        best_sentence, best_overlap = None, -1
        for text, _, _ in retrieved:
            for sent in _re.split(r'(?<=[.!?])\s+', text):
                s_words = set(_re.findall(r"\w+", sent.lower()))
                overlap = len(q_words & s_words)
                if overlap > best_overlap and len(sent.split()) > 3:
                    best_overlap, best_sentence = overlap, sent.strip()
        return best_sentence or retrieved[0][0][:300]

    print("Using extractive generation")

LLM generation path unavailable (OSError: We couldn't connect to 'https://huggingface.co' to load the files, and couldn't find them in the cached files.
Check your internet connection or see how to run the library in offline mode at 'https://huggingface.co/docs/transformers/installation#offline-mode'.)
Falling back to extractive generation (best-matching sentences, no download required).

Using extractive generation


## 6. Full pipeline — ask a question

In [6]:
def answer(question, top_k=TOP_K):
    retrieved = search(question, top_k=top_k)
    ans = generate_answer(question, retrieved)
    return {
        "question": question,
        "answer": ans,
        "sources": [f"{fname}#{cid} ({score:.3f})" for _, (fname, cid), score in retrieved],
    }

result = answer("What are the three stages of a RAG pipeline?")
print("Q:", result["question"])
print("A:", result["answer"])
print("Sources:", result["sources"])

Q: What are the three stages of a RAG pipeline?
A: A typical RAG pipeline has three stages.
Sources: ['sample.txt#0 (0.277)', 'sample.txt#1 (0.204)', 'sample.txt#2 (0.117)']


## 7. Test run — including an out-of-scope question
The last question ("capital of France") is deliberately not answerable from the document. A working RAG system should say so instead of hallucinating an answer from the base model's training data — this is the actual point of grounding generation in retrieval.

In [7]:
TEST_QUESTIONS = [
    "What is the main idea of the document?",
    "What are the three stages of a RAG pipeline?",
    "What is one advantage of RAG over fine-tuning?",
    "What is the capital of France?",
]

for q in TEST_QUESTIONS:
    r = answer(q)
    print("=" * 70)
    print("Q:", r["question"])
    print("A:", r["answer"])
    print("Sources:", r["sources"])
print("=" * 70)

Q: What is the main idea of the document?
A: The document does not contain this information.
Sources: []
Q: What are the three stages of a RAG pipeline?
A: A typical RAG pipeline has three stages.
Sources: ['sample.txt#0 (0.277)', 'sample.txt#1 (0.204)', 'sample.txt#2 (0.117)']
Q: What is one advantage of RAG over fine-tuning?
A: One key advantage of RAG over fine-tuning is that updating the knowledge base only requires re-indexing new documents, not retraining the model.
Sources: ['sample.txt#2 (0.281)', 'sample.txt#0 (0.120)', 'sample.txt#1 (0.056)']
Q: What is the capital of France?
A: The document does not contain this information.
Sources: []


## 8. Notes
- **Active mode in this run:** printed above at Stage 3-4 / Stage 7 (neural vs. TF-IDF, LLM vs. extractive) — depends on whether Hugging Face was reachable when this notebook was executed.
- **Production version:** `ingest.py`, `embed_store.py`, `rag_pipeline.py` (submitted separately) always use the neural path — sentence-transformers + FAISS + flan-t5-base — since a real deployment environment has internet access. This notebook adds the fallback purely so it's runnable and self-verifying in any environment.
- **Design rationale, architecture diagram, and extension ideas** (hybrid search, re-ranking, bigger generator) are in the accompanying report document.